# WATERWATCH AI 💧🤖
## Phase 2 & Phase 4: Data Cleaning, Preprocessing & Train-Test Splitting
**Role:** Member 1 – Data & ML Engineer  
**Project:** Explainable AI-Based Water Potability Prediction and Community Risk Monitoring System  

---

### Objectives & Core Principles
1. **Missing-Value Strategy:** Median imputation using Scikit-Learn `SimpleImputer` (preserves records without distortion from skewed tails).
2. **Outlier Investigation (IQR Method):** Identify and quantify outliers without deleting valid environmental contamination data.
3. **Data Leakage Prevention:** Feature-target separation and train-test splitting are performed **before** fitting imputers and scalers.
4. **Stratified Splitting (80/20):** Guarantees that training (2,620) and testing (656) partitions maintain the exact 61:39 class ratio.
5. **Scikit-Learn Pipeline:** Encapsulates median imputation and feature scaling (`StandardScaler`) for clean deployment and handover to Member 3.

### 1. Library Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Load raw dataset
df = pd.read_csv('../data/water_potability.csv')
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

### 2. Duplicate Detection & Data-Type Validation

In [ ]:
# Duplicate check
duplicates = df.duplicated().sum()
print(f"Duplicate records: {duplicates}")
assert duplicates == 0, "Duplicate rows detected!"

# Validate column types
print("\nColumn Data Types:")
print(df.dtypes)
assert all(df.drop(columns=['Potability']).dtypes == 'float64'), "All input features must be float64!"
assert df['Potability'].dtype == 'int64', "Target variable 'Potability' must be int64!"

### 3. Outlier Investigation Using the IQR Method
- Interquartile Range: $IQR = Q3 - Q1$
- Lower Bound: $Q1 - 1.5 \times IQR$
- Upper Bound: $Q3 + 1.5 \times IQR$
- **Policy:** Outliers are quantified and logged below, but **retained** because they reflect valid chemical pollution/risk extremes.

In [ ]:
features = df.columns.drop('Potability')
outlier_records = []

for col in features:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    outlier_records.append({
        'Feature': col,
        'Q1 (25%)': round(q1, 2),
        'Q3 (75%)': round(q3, 2),
        'IQR': round(iqr, 2),
        'Lower Bound': round(lower_bound, 2),
        'Upper Bound': round(upper_bound, 2),
        'Outlier Count': len(outliers),
        'Outlier %': round(len(outliers) / len(df) * 100, 2)
    })

outlier_summary_df = pd.DataFrame(outlier_records)
print("=== IQR OUTLIER SUMMARY TABLE ===")
display(outlier_summary_df)

### 4. Feature and Target Separation
- $X$ = 9 water-quality input features
- $y$ = `Potability` binary target variable (0 or 1)

In [ ]:
X = df.drop(columns=['Potability'])
y = df['Potability']

print(f"X shape (Input Features): {X.shape}")
print(f"y shape (Target Label):   {y.shape}")
print(f"Features: {list(X.columns)}")

### 5. Stratified Train-Test Splitting (80% Train, 20% Test)
- **Train size:** 80% (2,620 samples)
- **Test size:** 20% (656 samples)
- **Fixed random state:** `random_state=42` for exact reproducibility
- **Stratification:** `stratify=y` guarantees identical class proportions in both sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training partition shape: {X_train.shape}")
print(f"Testing partition shape:  {X_test.shape}\n")

train_class_ratio = y_train.value_counts(normalize=True) * 100
test_class_ratio = y_test.value_counts(normalize=True) * 100

split_dist = pd.DataFrame({
    'Train Count': y_train.value_counts(),
    'Train Ratio (%)': train_class_ratio.round(2),
    'Test Count': y_test.value_counts(),
    'Test Ratio (%)': test_class_ratio.round(2)
}, index=[0, 1])
split_dist.index.name = 'Potability Class'
print("=== STRATIFIED CLASS BALANCE VERIFICATION ===")
display(split_dist)

### 6. Scikit-Learn Preprocessing Pipeline (Median Imputation + Scaling)
- **Data Leakage Protection:**
  - `preprocessor.fit_transform(X_train)`: Learns the training medians and scaling parameters **only** from `X_train`.
  - `preprocessor.transform(X_test)`: Applies those learned training parameters to `X_test` with **zero data leakage**.

In [ ]:
# Define reusable preprocessing pipeline
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Fit strictly on X_train, then transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Verify zero missing values remain
assert np.isnan(X_train_processed).sum() == 0, "Missing values remain in training set!"
assert np.isnan(X_test_processed).sum() == 0, "Missing values remain in testing set!"

print("✓ Preprocessing Pipeline successfully fitted.")
print("✓ Missing values after imputation: 0")
print(f"✓ X_train_processed shape: {X_train_processed.shape}")
print(f"✓ X_test_processed shape:  {X_test_processed.shape}\n")

print("--- Learned Training Medians for Imputation ---")
for col, med in zip(X.columns, preprocessor.named_steps['imputer'].statistics_):
    print(f"  - {col:16s}: {med:.2f}")

### 7. Verification of Standardized Scale
Checking mean and standard deviation of scaled training features (expecting mean $\approx 0$ and std $\approx 1$):

In [ ]:
scaled_check_df = pd.DataFrame(X_train_processed, columns=X.columns)
print("=== SCALED TRAINING DATASET CHECK (Mean ≈ 0, Std ≈ 1) ===")
display(scaled_check_df.describe().T[['mean', 'std', 'min', 'max']].round(4))